## Ingestion

In [0]:
from pyspark.sql import functions as F

# Config
CATALOG = "olist_dw_project"
BRONZE_SCHEMA = "bronze"
VOLUME_PATH = "/Volumes/olist_dw_project/bronze/raw_files"

SOURCE_FILES = {
    "olist_orders_dataset.csv": "orders",
    "olist_order_items_dataset.csv": "order_items",
    "olist_order_payments_dataset.csv": "order_payments",
    "olist_customers_dataset.csv": "customers",
    "olist_products_dataset.csv": "products",
    "olist_sellers_dataset.csv": "sellers",
    "product_category_name_translation.csv": "product_category"
}

In [0]:
def ingest_to_bronze(file_name: str, table_name: str):
    source_path = f"{VOLUME_PATH}/{file_name}"
    target_table = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"

    df = spark.read.option("header", "true").option("inferSchema", "false").csv(source_path)
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true") .saveAsTable(target_table)
    
    row_count = df.count()
    print(f"Ingested {table_name}: {row_count:,} rows ingested -> {target_table}")
    
    return row_count

In [0]:
results = {}
for file_name, table_name in SOURCE_FILES.items():
    try:
        row_count = ingest_to_bronze(file_name, table_name)
        results[table_name] = row_count
    except Exception as e:
        print(f"Failed to ingest {file_name}: {e}")
        results[table_name] = "FAILED"

print("\nBronze Ingestion Summary")
for table, count in results.items():
    print(f"{table:30s} {count}")